# 20 · Scatter-seeded Eclat on the whole grid (~7 h)

Runs the seeded path, at eight threads and at one, across the grid above pairs, plus the dense datasets at k = 3, with the fastapriori 0.2.0 build and the `bench/exp_runners.py` harness. Defaults: 1 warm-up + 3 measured runs per cell, 1,500 s limit, 60 GiB cap, failures recorded, and larger k skipped once a cell fails. Never run two mining notebooks at the same time.

| Phase | Datasets | Support | k | Methods |
|---|---|---|---|---|
| grid | 14 sparse datasets, then Groceries | 1e-4 | 3-9 (T12.D5k and Groceries 3-6) | seeded |
| dense | chess, mushroom, connect, pumsb, accidents | 0.5 to 1e-3, high to low | 3 | seeded |
| gaps at 5e-4 | BMS-WebView-1, Kosarak, Online Retail; T12.D5k | 5e-4 | 3-9; 7-9 | seeded |
| gaps at 1e-4 | the same cells | 1e-4 | 3-9; 7-9 | seeded, Borgelt Eclat |
| tail | Groceries | 1e-4 | 7-9 | seeded |

The gap cells are the 24 of the 126 (18 datasets, k = 3-9) that the main grid did not attempt at 1e-4. Their comparators exist at 5e-4, or at 1e-4 through the Borgelt Eclat runs here. Groceries runs last because at k = 6 it already holds 44 GB of rules in Python.

Set `PHASES`, or `SPOT = True` for a five-minute check first. Outputs: `results/runs/<campaign>/seeded_grid.csv`, `seeded_dense.csv`, `seeded_gaps_5e4.csv`, `seeded_gaps_1e4.csv`.

`FA_MODE=replay` analyses the saved campaign (`vm_20261031` unless `FA_CAMPAIGN` says otherwise): no mining and no data needed.

In [ ]:
import os, sys, time
from pathlib import Path
BUNDLE = Path.cwd().resolve().parent     # notebooks run from <bundle>/notebooks
assert (BUNDLE / "bench" / "exp_runners.py").exists(), "start Jupyter in <bundle>/notebooks"
sys.path.insert(0, str(BUNDLE / "bench"))
REPLAY = os.environ.get("FA_MODE") == "replay"    # analyse the saved campaign: no mining, nothing overwritten
os.environ.setdefault("FA_CAMPAIGN", "vm_20261031" if REPLAY else "new_campaign")
import pandas as pd
import exp_runners as pr
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
print("campaign", pr.CAMPAIGN, "->", pr.RESULTS)
print("build", pr.build_id(), "| extension", pr.ext_path())

In [ ]:
SEEDED = ["hyb_seeded", "hyb_seeded_1t"]
FULL_K, TIMEOUT_S = [3, 4, 5, 6, 7, 8, 9], 1500
GRID = ["T12.I4.D5k.N1k.L1k", "T5.I2.D25k.N1k.L1k", "T3.I2.D100k.N50k.L50k", "BMS-WebView-2", "T15.I8.D50k.N1k.L1k.high",
        "T15.I8.D50k.N1k.L1k.low", "Retail (Belgian)", "T10.I4.D100k.N25k.L25k", "T30.I8.D50k.N5k.L5k", "Chainstore",
        "T20.I4.D1000k.N50k.L50k", "Instacart", "T30.I8.D2000k.N10k.L10k", "T25.I8.D5000k.N40k.L40k"]
GAPS = ["BMS-WebView-1", "Kosarak", "Online Retail"]
DENSE, DENSE_SUPPORTS = ["chess", "mushroom", "connect", "pumsb", "accidents"], [0.5, 0.1, 0.05, 0.01, 0.005, 1e-3]
SPOT = False
PHASES = [] if REPLAY else ["grid", "dense", "gaps_5e4", "gaps_1e4", "tail"]

def grid():
    for d in GRID:
        pr.run_grid("seeded_grid", [d], SEEDED, ks=[3, 4, 5, 6] if d.startswith("T12") else FULL_K, supports=[1e-4], timeout_s=TIMEOUT_S)
    pr.run_grid("seeded_grid", ["Groceries"], SEEDED, ks=[3, 4, 5, 6], supports=[1e-4], timeout_s=TIMEOUT_S)

def dense():
    for d in DENSE:                             # high to low support; a method that fails stops descending
        alive = list(SEEDED)
        for s in DENSE_SUPPORTS:
            if alive:
                pr.run_grid("seeded_dense", [d], alive, ks=[3], supports=[s], timeout_s=TIMEOUT_S)
                r = pr.load_results("seeded_dense")
                r = r[(r.dataset == d) & (r.min_support == s)]
                alive = [m for m in alive if (r[r.method == m].status == "ok").all()]

def gaps(s, methods, exp):
    pr.run_grid(exp, GAPS, methods, ks=FULL_K, supports=[s], timeout_s=TIMEOUT_S)
    pr.run_grid(exp, ["T12.I4.D5k.N1k.L1k"], methods, ks=[7, 8, 9], supports=[s], timeout_s=TIMEOUT_S)

def tail():
    pr.run_grid("seeded_grid", ["Groceries"], SEEDED, ks=[7, 8, 9], supports=[1e-4], timeout_s=TIMEOUT_S)

RUN = dict(grid=grid, dense=dense, gaps_5e4=lambda: gaps(5e-4, SEEDED, "seeded_gaps_5e4"),
           gaps_1e4=lambda: gaps(1e-4, SEEDED + ["borgelt_eclat"], "seeded_gaps_1e4"), tail=tail)

In [ ]:
if SPOT and not REPLAY:
    pr.run_grid("spot", ["T10.I4.D100k.N25k.L25k", "Instacart"], ["hyb"] + SEEDED, ks=[3], supports=[1e-4], repeats=1, timeout_s=300)
    pr.run_grid("spot", ["Groceries"], ["hyb_seeded"], ks=[5, 6], supports=[1e-4], repeats=1, warmup=0, timeout_s=900)
    display(pr.load_results("spot")[["dataset", "method", "k", "warmup", "status", "wall_s", "peak_rss_kb", "n_itemsets"]])

In [ ]:
for name in PHASES:
    t0 = time.time()
    RUN[name]()
    print(name, f"{(time.time() - t0) / 60:.1f} min")

In [ ]:
for exp in ("seeded_grid", "seeded_dense", "seeded_gaps_5e4", "seeded_gaps_1e4"):
    r = pr.load_results(exp)
    if len(r):
        print(exp, r[r.warmup].status.value_counts().to_dict())   # failures and skips end at the warm-up
        print(r[r.status.isin(["oom", "timeout"])][["dataset", "method", "k", "min_support", "status", "peak_rss_kb"]].to_string(index=False))
        display(pr.summarize(r).pivot_table(index=["dataset", "min_support", "k"], columns="method", values="wall_s").round(3))